# Cotracking prototype (zarr input)

Sections 1-3 are the cells from `organelle_tracker_testing_20251219.ipynb`, copied **unchanged** (segmentation, `filter_dense_blobs`, Kalman linking, net-displacement filter). Only the input (now the zarr instead of the ND2) and the particle-track source are set in the config cell. Section 4 onward is new and only reads the resulting DataFrames.

## 0. Config

In [ ]:
from pathlib import Path
import sys

# ---- edit these ----
RAW_ROOT  = Path("/Users/kelpschdj/Documents/DataTecnica/TTU/Data/Sphere/220725_i11w-hT-M33-I76_sg1035_d10sphere/raw_data")
MOVIE     = "sg100_Well5_1018"          # change to run another movie
ZARR_PATH = RAW_ROOT / f"{MOVIE}.zarr"
PARTICLE_CSV = Path("/Users/kelpschdj/Documents/DataTecnica/TTU/TrainTracks/sg100_Well5_1018_valid_tracks.csv")  # frame,x,y,particle
COTRACK_DIR  = Path("/Users/kelpschdj/Documents/DataTecnica/TTU/TrainTracks/cotracking")
sys.path.insert(0, str(COTRACK_DIR))


## 1. Load image from zarr

Same array layout the notebook got from `nd2.imread`: `(T, C, Y, X)` = (33, 4, 2048, 2048), uint16. Channels: 0 = Halo-TDP43 (particles), 1 = lysosome, 2 = mitochondria, 3 = sg/BFP. Read as a plain array with no scale transform, so pixel units are identical to the particle-track CSV.

In [ ]:
import numpy as np
import zarr
root = zarr.open_group(str(ZARR_PATH), mode="r")
img_array = np.asarray(root["s0"][:])      # (T, C, Y, X), loaded into memory (~1.1 GB)
print(ZARR_PATH.name, img_array.shape, img_array.dtype)
assert img_array.ndim == 4 and img_array.shape[1] >= 3, "expected (T, C, Y, X) with >= 3 channels"

## 2. Segmentation + centroids (unchanged)

In [ ]:
from scipy.ndimage import gaussian_laplace
from skimage import filters, morphology
from scipy import ndimage as ndi
from scipy.stats import norm
import numpy as np

def generate_lysosome_mask(
    frame,
    intensity_scaling_param=[3, 19],
    min_lysosome_area=15,
    blur_sigma=1,
    log_params=[(5, 0.09), (2.5, 0.07), (1, 0.01)],
    vesselness_sigma=[1],
    vesselness_cutoff=0.15):

    # Normalize to normal distribution
    m, s = norm.fit(frame.flatten())
    stretch_min = max(m - intensity_scaling_param[0] * s, frame.min())
    stretch_max = min(m + intensity_scaling_param[1] * s, frame.max())
    frame = np.clip(frame, stretch_min, stretch_max)
    image_norm = (frame - stretch_min) / (stretch_max - stretch_min)

    # Blur frame
    blurred = filters.gaussian(image_norm, sigma=blur_sigma)

    # LoG blob detection
    log_masks = []
    for sigma, cutoff in log_params:
        log_img = -1 * (sigma**2) * gaussian_laplace(blurred, sigma=sigma)
        log_masks.append(log_img > cutoff)
    log_mask = np.logical_or.reduce(log_masks)

    # Frangi vesselness
    vesselness = filters.frangi(blurred, sigmas=vesselness_sigma, black_ridges=False) > vesselness_cutoff

    # Combine and postprocess
    combined = np.logical_or(log_mask, vesselness)
    filled = ndi.binary_fill_holes(combined)
    cleaned = morphology.remove_small_objects(filled, min_size=min_lysosome_area)

    return cleaned.astype(np.uint8)

def generate_mitochondria_mask(
    frame,
    intensity_scaling_param=(3.5, 15),
    min_mitochondria_area=10,
    blur_sigma=1,
    log_params=((5, 0.09), (2.5, 0.07), (1, 0.01)),
    vesselness_sigma=(1.5,),
    vesselness_cutoff=0.16
):
    # Normalize using Gaussian fit
    m, s = norm.fit(frame.flatten())
    stretch_min = max(m - intensity_scaling_param[0] * s, frame.min())
    stretch_max = min(m + intensity_scaling_param[1] * s, frame.max())
    clipped = np.clip(frame, stretch_min, stretch_max)
    image_norm = (clipped - stretch_min) / (stretch_max - stretch_min + 1e-8)  # prevent div-by-zero

    # Blur frame
    blurred = filters.gaussian(image_norm, sigma=blur_sigma)

    # Multi-scale LoG filtering
    log_masks = []
    for sigma, cutoff in log_params:
        log_response = -1 * (sigma ** 2) * gaussian_laplace(blurred, sigma=sigma)
        log_masks.append(log_response > cutoff)

    log_mask = np.logical_or.reduce(log_masks)

    # Vesselness (Frangi)
    vesselness = filters.frangi(blurred, sigmas=vesselness_sigma, black_ridges=False)
    vessel_mask = vesselness > vesselness_cutoff

    # Combine masks
    combined = np.logical_or(log_mask, vessel_mask)

    # Remove small objects
    mitochondria_mask = morphology.remove_small_objects(combined, min_size=min_mitochondria_area)

    return mitochondria_mask

In [ ]:
import dask.array as da
import dask
from dask.diagnostics import ProgressBar

# Set channel indices
lysosome_channel = 1
mitochondria_channel = 2

# Convert to Dask array (chunk by time)
img_dask = da.from_array(img_array, chunks=(1, -1, -1, -1))  # (T, C, Y, X)

def process_frame(frame_4d, lysosome_channel, mitochondria_channel):
    lyso_frame = frame_4d[lysosome_channel].astype(np.float32)
    mito_frame = frame_4d[mitochondria_channel].astype(np.float32)

    lyso_mask = generate_lysosome_mask(lyso_frame)
    mito_mask = generate_mitochondria_mask(mito_frame)

    return lyso_mask, mito_mask

# Wrap into delayed loop
delayed_results = [
    dask.delayed(process_frame)(img_dask[t], lysosome_channel, mitochondria_channel)
    for t in range(img_dask.shape[0])
]

# Unpack and compute
with ProgressBar():
    results = dask.compute(*delayed_results)  # List of (lyso_mask, mito_mask) tuples

# Unzip and stack
lyso_masks, mito_masks = zip(*results)
lysosome_mask_stack = np.stack(lyso_masks, axis=0)
mitochondria_mask_stack = np.stack(mito_masks, axis=0)

In [ ]:
import pandas as pd
from skimage.measure import label, regionprops_table

def mask_to_centroids(mask, frame_idx, organelle_type):
    """
    mask: 2D binary array
    frame_idx: int frame number
    organelle_type: "lysosome" or "mitochondria"
    """
    mask_bool = mask.astype(bool)
    labels = label(mask_bool)

    if labels.max() == 0:
        # No objects in this frame
        return pd.DataFrame(columns=["frame", "x", "y", "area", "label", "organelle"])

    props = regionprops_table(
        labels,
        properties=("label", "area", "centroid")
    )
    df = pd.DataFrame(props)

    # regionprops gives centroid-0 = row (y), centroid-1 = col (x)
    df.rename(columns={"centroid-0": "y", "centroid-1": "x"}, inplace=True)
    df["frame"] = frame_idx
    df["organelle"] = organelle_type

    return df[["frame", "x", "y", "area", "label", "organelle"]]

lys_dfs = []
mito_dfs = []

T = lysosome_mask_stack.shape[0]

for t in range(T):
    lyso_mask_frame = lysosome_mask_stack[t]
    mito_mask_frame = mitochondria_mask_stack[t]

    df_lys = mask_to_centroids(lyso_mask_frame, frame_idx=t, organelle_type="lysosome")
    df_mito = mask_to_centroids(mito_mask_frame, frame_idx=t, organelle_type="mitochondria")

    lys_dfs.append(df_lys)
    mito_dfs.append(df_mito)

# Combine all frames
lys_blobs = pd.concat(lys_dfs, ignore_index=True)
mito_blobs = pd.concat(mito_dfs, ignore_index=True)

def filter_dense_blobs(blob_df, bin_size=5, blob_filter=10):
    """
    Filters out blobs that appear too frequently within a very small local neighborhood.

    Parameters:
        blob_df (pd.DataFrame): DataFrame with 'x' and 'y' columns of blob coordinates.
        bin_size (int): Pixel binning size for rounding positions.
        blob_filter (int): Maximum allowed count of blobs in a bin before filtering them out.

    Returns:
        pd.DataFrame: Filtered DataFrame with blobs below the density threshold.
    """
    rounded = blob_df.copy()
    rounded['x_rounded'] = (rounded['x'] // bin_size * bin_size).astype(int)
    rounded['y_rounded'] = (rounded['y'] // bin_size * bin_size).astype(int)

    counts = rounded.groupby(['x_rounded', 'y_rounded']).size().reset_index(name='count')
    merged = rounded.merge(counts, on=['x_rounded', 'y_rounded'])

    filtered_blobs = merged[merged['count'] < blob_filter].reset_index(drop=True)

    print(f"Puncta filtering complete: {len(filtered_blobs)} puncta remain")
    return filtered_blobs

lys_blobs = filter_dense_blobs(lys_blobs, blob_filter=5)
mito_blobs = filter_dense_blobs(mito_blobs, blob_filter=5)

## 3. Kalman linking + track filter (unchanged)

In [ ]:
from filterpy.kalman import KalmanFilter
from scipy.spatial import cKDTree

class KalmanTrack:
    def __init__(self, initial_detection, particle_id, frame, max_missed=5):
        x, y = initial_detection
        self.kf = KalmanFilter(dim_x=4, dim_z=2)
        dt = 1
        self.kf.F = np.array([[1, 0, dt, 0],
                              [0, 1, 0, dt],
                              [0, 0, 1, 0],
                              [0, 0, 0, 1]])
        self.kf.H = np.array([[1, 0, 0, 0],
                              [0, 1, 0, 0]])
        self.kf.R *= 5
        self.kf.P *= 1000
        self.kf.Q = np.eye(4) * 0.1
        self.kf.x = np.array([x, y, 0, 0])

        self.id = particle_id
        self.history = [{'frame': frame, 'x': x, 'y': y, 'particle': self.id}]
        self.missed = 0
        self.max_missed = max_missed

    def predict(self):
        self.kf.predict()
        return self.kf.x[:2]

    def update(self, detection, frame):
        self.kf.update(np.array(detection))
        x, y = self.kf.x[:2]
        self.history.append({'frame': frame, 'x': x, 'y': y, 'particle': self.id})
        self.missed = 0

    def mark_missed(self):
        self.missed += 1

    def is_dead(self):
        return self.missed > self.max_missed

def kalman_track_blobs(all_blobs, max_distance=15, max_missed=5):
    active_tracks = []
    finished_tracks = []
    next_id = 0

    for frame, detections in all_blobs.groupby('frame'):
        detected_positions = detections[['x', 'y']].to_numpy()

        # Predict positions for all current tracks
        predictions = [track.predict() for track in active_tracks]
        assigned = set()

        if len(predictions) > 0 and len(detected_positions) > 0:
            tree = cKDTree(detected_positions)
            for i, pred in enumerate(predictions):
                dist, idx = tree.query(pred, distance_upper_bound=max_distance)
                if idx != len(detected_positions) and idx not in assigned:
                    active_tracks[i].update(detected_positions[idx], frame)
                    assigned.add(idx)
                else:
                    active_tracks[i].mark_missed()
        else:
            for track in active_tracks:
                track.mark_missed()

        # Start new tracks for unassigned detections
        for i, pos in enumerate(detected_positions):
            if i not in assigned:
                new_track = KalmanTrack(pos, next_id, frame, max_missed)
                active_tracks.append(new_track)
                next_id += 1

        # Remove dead tracks
        still_alive = []
        for track in active_tracks:
            if track.is_dead():
                finished_tracks.append(track)
            else:
                still_alive.append(track)
        active_tracks = still_alive

    # Add remaining active tracks to finished
    finished_tracks.extend(active_tracks)

    # Combine history
    all_tracks = [row for track in finished_tracks for row in track.history]        
    all_tracks = pd.DataFrame(all_tracks)
    print(f"Particle tracking complete: There are {len(all_tracks.groupby('particle'))} unique particle tracks")
    return all_tracks


In [ ]:
tracks_lys = kalman_track_blobs(lys_blobs, max_distance=25, max_missed=4)
tracks_mito = kalman_track_blobs(mito_blobs, max_distance=25, max_missed=3)

In [ ]:
import trackpy as tp 

def filter_tracks_by_net_displacement(tracks_df, displacement_threshold=50, min_frames=10):
    """
    Filter tracks by net displacement after removing short tracks.

    Parameters:
        tracks_df (pd.DataFrame): DataFrame with ['particle', 'x', 'y'].
        displacement_threshold (float): Minimum net displacement required to keep a track.
        min_frames (int): Minimum number of frames a track must appear in.

    Returns:
        pd.DataFrame: Filtered tracking DataFrame.
    """
    # Remove short tracks
    filtered_tracks = tp.filter_stubs(tracks_df, threshold=min_frames).reset_index(drop=True)
    print(f"Track stub removal complete: {len(filtered_tracks.groupby('particle'))} unique particle tracks")

    # Compute net displacement
    displacements = []
    for pid, group in filtered_tracks.groupby('particle'):
        start = group.iloc[0][['x', 'y']].values
        end = group.iloc[-1][['x', 'y']].values
        net_disp = np.linalg.norm(end - start)
        displacements.append({'particle': pid, 'net_displacement': net_disp})

    displacement_df = pd.DataFrame(displacements)

    # Filter by displacement
    valid_particles = displacement_df.loc[displacement_df['net_displacement'] > displacement_threshold, 'particle']
    print(f"Track filtering complete: {len(valid_particles)} unique particle tracks")

    return filtered_tracks[filtered_tracks['particle'].isin(valid_particles)].reset_index(drop=True)

tracks_lys = filter_tracks_by_net_displacement(tracks_lys)
tracks_mito = filter_tracks_by_net_displacement(tracks_mito)

## 4. Particle tracks for 1018

Same columns the organelle tracks have (`frame, x, y, particle`).

In [ ]:
import pandas as pd
particle_tracks = pd.read_csv(PARTICLE_CSV, encoding="utf-8-sig")[["frame", "x", "y", "particle"]]
print(particle_tracks.particle.nunique(), "particle tracks,", len(particle_tracks), "detections")
print("lysosome tracks:", tracks_lys.particle.nunique(), "| mitochondria tracks:", tracks_mito.particle.nunique())

## 5. Cotracking parameters

Distances are **pixels**. Start with `contact_px` near the lysosome radius (check the median `area` in `lys_blobs`; radius = sqrt(area/pi)). Sweep it in section 8.

In [ ]:
from cotracking import (find_episodes, particle_steps, segment_summary,
                        paired_during_vs_outside, shifted_null, compare_to_null)

EP = dict(contact_px=10.0, min_contact_frames=3, max_gap=1, stationary_px=5.0, cos_thresh=0.5)

import numpy as np
print("median lysosome radius px :", np.sqrt(lys_blobs.area.median()/np.pi).round(1))
print("median mito 'radius' px   :", np.sqrt(mito_blobs.area.median()/np.pi).round(1))

## 6. Episodes (lysosome and mitochondria run separately)

In [ ]:
ep_lys  = find_episodes(particle_tracks, tracks_lys,  **EP)
ep_mito = find_episodes(particle_tracks, tracks_mito, **EP)
print("lysosome episodes:", len(ep_lys), "| mito episodes:", len(ep_mito))
ep_lys

In [ ]:
ep_mito

## 7. Particle behaviour during vs outside cotracking (paired within track)

In [ ]:
steps_lys = particle_steps(particle_tracks, ep_lys)
print(steps_lys.groupby("state").speed.agg(["size", "mean", "median"]))
paired_lys = paired_during_vs_outside(steps_lys)
print(paired_lys)
segment_summary(steps_lys)

## 8. Null model

Translates the whole organelle track set by a random 40-120 px vector (`local`, preserves neighbourhood density) and recomputes episodes. With only one movie and a handful of particles, treat this as a sanity check; pool across movies for statistics.

In [ ]:
null_lys = shifted_null(particle_tracks, tracks_lys, n_perm=200, mode="local",
                        shift_range=(40, 120), seed=0, **EP)
compare_to_null(particle_tracks, ep_lys, null_lys)

In [ ]:
null_mito = shifted_null(particle_tracks, tracks_mito, n_perm=200, mode="local",
                         shift_range=(40, 120), seed=0, **EP)
compare_to_null(particle_tracks, ep_mito, null_mito)

## 9. Sensitivity to contact distance

In [ ]:
rows = []
for cpx in [5, 8, 10, 15, 20, 30]:
    kw = {**EP, "contact_px": cpx}
    e = find_episodes(particle_tracks, tracks_lys, **kw)
    n = shifted_null(particle_tracks, tracks_lys, n_perm=50, seed=0, **kw)
    rows.append((cpx, len(e), n.n_episodes.mean(), e.episode_class.value_counts().to_dict()))
pd.DataFrame(rows, columns=["contact_px", "observed_episodes", "null_mean_episodes", "classes"])

## 10. Look at it in napari

Episodes are drawn as lines from the particle to the organelle at each contact frame.

In [ ]:
import napari
viewer = napari.Viewer()
viewer.add_image(img_array[:, 1], name="LYSOSOME", colormap="red", blending="additive")
viewer.add_image(img_array[:, 2], name="MITOCHONDRIA", colormap="green", blending="additive")
viewer.add_image(img_array[:, 0], name="Halo-TDP43", colormap="magenta", blending="additive")
viewer.add_tracks(tracks_lys[["particle","frame","y","x"]].to_numpy(),  name="LYSOSOME TRACKS")
viewer.add_tracks(tracks_mito[["particle","frame","y","x"]].to_numpy(), name="MITO TRACKS")
pt = viewer.add_tracks(particle_tracks[["particle","frame","y","x"]].to_numpy(), name="PARTICLE TRACKS")
pt.display_id = True

# particle positions during episodes
pts = []
for _, r in ep_lys.iterrows():
    g = particle_tracks[(particle_tracks.particle == r.particle) & particle_tracks.frame.between(r.start, r.end)]
    pts += g[["frame","y","x"]].values.tolist()
if pts:
    viewer.add_points(np.array(pts), name="lyso cotracking", size=30, face_color="transparent",
                      border_color="yellow", border_width=0.1)

## 11. Save

In [ ]:
out = COTRACK_DIR / MOVIE
out.mkdir(exist_ok=True)
ep_lys.to_csv(out / "episodes_lysosome.csv", index=False)
ep_mito.to_csv(out / "episodes_mitochondria.csv", index=False)
steps_lys.to_csv(out / "particle_steps_lysosome.csv", index=False)
tracks_lys.to_csv(out / "tracks_lysosome.csv", index=False)
tracks_mito.to_csv(out / "tracks_mitochondria.csv", index=False)